# **04 - SelfCheck-MQAG (QA)**

In [ ]:
# Restart after running this cell.
!pip install -q "transformers==4.36.2" selfcheckgpt sentencepiece "accelerate>=0.26,<1.0" "bitsandbytes>=0.43.0"
!python -m spacy download en_core_web_sm -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.7.0 requires tokenizers>=0.19, but you have tokenizers 0.15.2 which is incompatible.
sentence-transformers 5.7.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to 

In [ ]:
import os, sys
try:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_DIR = "/content/drive/MyDrive/selfcheckgpt_replication_results"
except ImportError:                      # running outside Colab
    RESULTS_DIR = "./selfcheckgpt_results"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.environ["SC_RESULTS_DIR"] = RESULTS_DIR
if RESULTS_DIR not in sys.path: sys.path.insert(0, RESULTS_DIR)
try:
    from sc_common import *            # shared config, dataset, metrics, cache helpers 
except ModuleNotFoundError:
    raise FileNotFoundError(f"Upload sc_common.py into {RESULTS_DIR} first.")
describe_device(); ram("after setup")

Mounted at /content/drive


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/evaluation-00000-of-00001-e91191b8f(…):   0%|          | 0.00/2.56M [00:00<?, ?B/s]

Generating evaluation split:   0%|          | 0/238 [00:00<?, ? examples/s]

Using device: cuda
Tesla T4 - 14.6 GB
Results / caches -> /content/drive/MyDrive/selfcheckgpt_replication_results
[RAM after setup] this process 0.8 GB | system used 1.7 / 12.7 GB


In [2]:
from selfcheckgpt.modeling_selfcheck import (SelfCheckMQAG, method_simple_counting, method_vanilla_bayes,
                                             method_bayes_with_alpha, answerability_scoring)
from selfcheckgpt.modeling_mqag import question_generation_sentence_level, answering
mqag = SelfCheckMQAG(device=device)

def _amp():
    return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=bool(MQAG_FP16) and device.type == "cuda")

@torch.no_grad()
def mqag_passage_cache(sentences, passage, samples, num_q=MQAG_NUM_QUESTIONS, seed=None):
    # same computation as SelfCheckMQAG.predict, but keeps the answer distributions so that any scoring rule can be applied afterwards
    if seed is not None: torch.manual_seed(seed)
    per_sentence = []
    for sentence in sentences:
        questions = question_generation_sentence_level(mqag.g1_model, mqag.g1_tokenizer, mqag.g2_model, mqag.g2_tokenizer,
                                                       sentence, passage, num_q, device)
        items = []
        for q in questions:
            with _amp():
                prob = answering(mqag.a_model, mqag.a_tokenizer, q["question"], q["options"], passage, 4096, device)
                u = answerability_scoring(mqag.u_model, mqag.u_tokenizer, q["question"], passage, 4096, device)
                prob_s, u_s = np.zeros((len(samples), 4)), np.zeros(len(samples))
                for si, sp in enumerate(samples):
                    prob_s[si] = answering(mqag.a_model, mqag.a_tokenizer, q["question"], q["options"], sp, 4096, device)
                    u_s[si] = answerability_scoring(mqag.u_model, mqag.u_tokenizer, q["question"], sp, 4096, device)
            items.append({"prob": np.asarray(prob, dtype=np.float64), "u": float(u),
                          "prob_s": prob_s.astype(np.float64), "u_s": u_s.astype(np.float64)})
        per_sentence.append(items)
    return per_sentence

ram("after loading MQAG models")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

... (21 outputs left out)

special_tokens_map.json:   0%|          | 0.00/957 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.74G [00:00<?, ?B/s]

SelfCheck-MQAG initialized to device cuda
[RAM after loading MQAG models] this process 10.8 GB | system used 11.5 / 12.7 GB


In [ ]:
if VERIFY_AGAINST_PACKAGE:    
    r = rows[MQAG_IDS[0]]; sent, samp = r["gpt3_sentences"][:1], r["gpt3_text_samples"][:3]
    torch.manual_seed(123)
    ref = mqag.predict(sentences=sent, passage=r["gpt3_text"], sampled_passages=samp, num_questions_per_sent=MQAG_NUM_QUESTIONS,
                       scoring_method="bayes_with_alpha", beta1=0.8, beta2=0.8)
    _fp16, MQAG_FP16 = MQAG_FP16, False
    mine, _ = mqag_scores({0: mqag_passage_cache(sent, r["gpt3_text"], samp, seed=123)})
    MQAG_FP16 = _fp16
    check_close("MQAG cached scoring vs package predict()", mine[0], ref, 1e-3)

t0 = time.time()
QCACHE["samples"] = run_cached("mqag_samples",
                               lambda i: mqag_passage_cache(rows[i]["gpt3_sentences"], rows[i]["gpt3_text"], get_samples(i), seed=SEED + i),
                               MQAG_IDS, "SelfCheck-MQAG", save_every=3)
print(f"done in {(time.time() - t0) / 60:.1f} min")

sc, empty = mqag_scores(QCACHE["samples"])
print(f"sentences without any valid generated question (scored 0.5): {empty}")
register_main("SelfCheck-QA", sc)

# three scoring rules on the same questions
for label, method in (("QA SimpleCount", "counting"), ("QA +Bayes", "bayes"), ("QA +Bayes+alpha", "bayes_with_alpha")):
    RES_QA[label] = evaluate(mqag_scores(QCACHE["samples"], method=method)[0])

if RUN_WIKIBIO_EVIDENCE:
    QCACHE["wiki"] = run_cached("mqag_wiki",
                                lambda i: mqag_passage_cache(rows[i]["gpt3_sentences"], rows[i]["gpt3_text"], get_samples(i, "wiki"), seed=SEED + i),
                                MQAG_IDS, "MQAG (WikiBio)", save_every=5)
    RES_WIKI["WikiBio+QA"] = evaluate(mqag_scores(QCACHE["wiki"])[0])


[OK] MQAG cached scoring vs package predict(): max |diff| = 0.00e+00 (tol 0.001)
SelfCheck-MQAG: 0 cached, 20 to compute


SelfCheck-MQAG:   0%|          | 0/20 [00:00<?, ?it/s]

done in 143.2 min
sentences without any valid generated question (scored 0.5): 0
SelfCheck-QA {'NonFact': np.float64(82.43), 'NonFact*': np.float64(36.78), 'Factual': np.float64(41.62), 'Pearson': np.float64(68.4), 'Spearman': np.float64(69.48), 'n_passages': 20}
MQAG (WikiBio): 0 cached, 20 to compute


MQAG (WikiBio):   0%|          | 0/20 [00:00<?, ?it/s]